# pandapipes component architecture: from `build_system_matrix` to `register_*`

This is a developer-facing tutorial, not an end-user "how to build a network" tutorial - it
explains how a component (`ExtGrid`, `Pipe`, `CircPump`, ...) actually turns into rows and columns
of the sparse Newton system that `pipeflow()` solves. `ExtGrid` is used as the running example
throughout, because it happens to touch nearly every piece of the machinery: both PIT-write
phases, all three `PitWriteMode`s, both `EqWriteMode`s that matter in practice, and a
cross-component coordination pattern with `CircPump` that's worth understanding in its own right.


## 1. Two arrays, one net: `node_pit` and `branch_pit`

Before any equations exist, pandapipes builds two big NumPy arrays per solve:

- `node_pit` - one row per junction, columns defined in `idx_node.py`'s `IdxNode` (e.g.
  `IdxNode.PINIT`, `IdxNode.NODE_TYPE`, ...)
- `branch_pit` - one row per branch element (pipe, valve, pump, ...), columns defined in
  `idx_branch.py`'s `IdxBranch`

Every component reads and writes these through **named columns**, never raw integers - e.g.
`node_pit[:, IdxNode.PINIT]`, not `node_pit[:, 10]`. Let's build a tiny network and look at the
raw array to make this concrete.

In [ ]:
import numpy as np
import pandapipes
from pandapipes.idx_node import IdxNode

net = pandapipes.create_empty_network(fluid="water")
j0 = pandapipes.create_junction(net, pn_bar=5, tfluid_k=285.15)
j1 = pandapipes.create_junction(net, pn_bar=5, tfluid_k=285.15)
pandapipes.create_pipe_from_parameters(net, j0, j1, length_km=1.0, k_mm=0.1, inner_diameter_mm=100)
pandapipes.create_ext_grid(net, j0, p_bar=5, t_k=285.15, type="pt")
pandapipes.create_sink(net, j1, mdot_kg_per_s=1.0)

pandapipes.pipeflow(net)

node_pit = net["_pit"]["node"]
print("node_pit shape:", node_pit.shape, " (node_cols =", IdxNode.node_cols, ")")
print("PINIT column:     ", node_pit[:, IdxNode.PINIT])
print("NODE_TYPE column: ", node_pit[:, IdxNode.NODE_TYPE], " (IdxNode.P =", IdxNode.P, ")")


`IdxNode`/`IdxBranch` are plain integer constants (via a small `IndexMeta` metaclass) - they
don't own any behavior, they're purely a naming scheme for array columns.

Getting a component's data into these arrays, and then turning that data into equations, happens
in **two separate phases**, each with its own registration mechanism.

## 2. Phase 1 - filling the PIT: `register_pit_node_entries`

Before the Newton loop starts, every component's `register_pit_*_entries` classmethod runs once,
writing its input data into `node_pit`/`branch_pit`. This is where `net.ext_grid.p_bar` becomes
`node_pit[some_row, IdxNode.PINIT]`.

Components don't write to the array directly. They build a `PitEntries` object (COO-style: rows,
cols, data) and hand it to a `PitRegistry` via `registry.add(...)` or `registry.add_override(...)`.
The registry defers the actual array write until every component has registered, then applies them
all via `PitRegistry.apply()` (`pf/system_index.py`).

### `ExtGrid.register_pit_node_entries` (`component_models/ext_grid_component.py`)

```python
@classmethod
def register_pit_node_entries(cls, net, node_pit, registry) -> None:
    ext_grids = net[cls.table_name()]
    ext_grids = ext_grids[ext_grids[cls.active_identifier()].values]
    if not len(ext_grids):
        return

    junction = ext_grids[cls.get_node_col()].values
    types = ext_grids.type.values
    junction_lookup = get_lookup(net, "node", "index")[cls.get_connected_node_type().table_name()]
    mask_p = np.isin(types, ["p", "pt"])
    mask_t = np.isin(types, ["t", "pt"])
    index_p = junction_lookup[junction[mask_p]]
    index_t = junction_lookup[junction[mask_t]]

    registry.add_override(PitEntries(*build_pit_entries(
        index_p,
        [IdxNode.PINIT, IdxNode.NODE_TYPE],
        [ext_grids.p_bar.values[mask_p], float(IdxNode.P)],
    ), mode=PitWriteMode.MEAN))
    registry.add_override(PitEntries(*build_pit_entries(
        index_t,
        [IdxNode.TINIT, IdxNode.NODE_TYPE_T],
        [ext_grids.t_k.values[mask_t], float(IdxNode.T)],
    ), mode=PitWriteMode.MEAN))
    registry.add_override(PitEntries(*build_pit_entries(
        index_p,
        [IdxNode.VAR_MASS_SLACK],
        [1.]),
        mode=PitWriteMode.UNIQUE))
```

`junction_lookup` translates a junction-table index into a `node_pit` row index (junctions from
different tables all share one `node_pit`, so this mapping is needed even in the simple case).
`build_pit_entries(rows, cols, data)` just fans `rows` out against every `(col, data)` pair to
build the flat COO arrays `PitEntries` wants.

### `PitWriteMode` - what happens when a cell gets written more than once

```python
class PitWriteMode(str, Enum):
    UNIQUE   = "unique"    # exclusive write - conflict check on registration, direct assignment
    ADDITIVE = "additive"  # values accumulated with np.add.at
    MEAN     = "mean"      # mean of all values written to the same (row, col) position
```

- **`MEAN`** for `PINIT`/`NODE_TYPE`: if two ext_grids sit at the same junction with the same
  `p_bar`, both write the same value - averaging is a no-op. If they *disagree*, `MEAN` degrades
  gracefully (averages them) instead of raising - a deliberate choice for this cell, not an
  oversight.
- **`UNIQUE`** for `VAR_MASS_SLACK`: this is a plain "yes/no" flag ("is there a *real* ext_grid at
  this junction"), not a count to average or accumulate - `UNIQUE` matches that intent directly
  (single, direct write).
- **`ADDITIVE`** is for genuine counters/sums.

`add()` vs. `add_override()` (both on `PitRegistry`) determine write **order**, not write
semantics: everything in `add()` (the "normal" bucket) is applied before everything in
`add_override()` (the "overrides" bucket), so overrides win when both target the same
`(row, col)` under `UNIQUE`. `ExtGrid` uses `add_override` throughout because a junction's
`PINIT`/`NODE_TYPE` default (plain `Junction.register_pit_node_entries`, written via `add()`) must
yield to whatever an ext_grid dictates for that node.

Let's see `MEAN` in action - two ext_grids at the same junction, splitting one sink's demand:

In [ ]:
net2 = pandapipes.create_empty_network(fluid="water")
j0 = pandapipes.create_junction(net2, pn_bar=5, tfluid_k=285.15)
pandapipes.create_ext_grid(net2, j0, p_bar=5, t_k=285.15, type="pt")
pandapipes.create_ext_grid(net2, j0, p_bar=5, t_k=285.15, type="pt")
pandapipes.create_sink(net2, j0, mdot_kg_per_s=5.0)

pandapipes.pipeflow(net2)
print(net2.res_ext_grid)


> **Sidebar - a counter that turned out to be dead weight.** `ExtGrid` used to also write
> `IdxNode.EXT_GRID_OCCURENCE` (`ADDITIVE`, counting how many ext_grid rows share a junction), so
> `extract_results` could divide a shared `MDOTSLACKINIT` value by that count to report each
> ext_grid's individual share. Once `register_hydraulic_equations` was rewritten to register once
> **per ext_grid row** instead of once per de-duplicated junction (see below), the linear system
> itself produces each ext_grid's own share directly - the count, and the division, became
> unnecessary and were deleted along with `EXT_GRID_OCCURENCE`/`EXT_GRID_OCCURENCE_T`. Worth
> remembering next time you touch code like this: a "helper" column can silently outlive the one
> thing that needed it - check whether every column you're about to preserve is still actually
> *read* anywhere, not just written.

## 3. `HydraulicSystemIndex` - where a variable "lives" in the big matrix

Once every component's PIT data is in place (and, for hydraulics, connectivity has been resolved
and the pit reduced to only the active rows), the Newton loop starts. Every iteration,
`solve_hydraulics` (`pf/calculation.py`) builds a `HydraulicSystemIndex` for the current pit:

```python
class HydraulicSystemIndex(BaseSystemIndex):
    """
    Layout (columns = rows in square system):
        0 .. len_n-1             PINIT / NODE          pressure / node mass-balance
        len_n .. len_n+len_b-1   MDOTINIT / BRANCH     mass flow / branch momentum
        len_n+len_b .. ...       MDOTSLACKINIT / SLACK  slack-mass variables (P-type nodes)
    """
    def __init__(self, node_pit, branch_pit):
        self.slack_nodes = np.where(node_pit[:, IdxNode.NODE_TYPE] == IdxNode.P)[0].astype(np.int32)
        len_n, len_b, len_s = len(node_pit), len(branch_pit), len(self.slack_nodes)
        slack_vals = np.arange(len_s, dtype=np.int32) + len_n + len_b

        self._register(HydVarEq.PINIT,    np.arange(len_n))
        self._register(HydVarEq.MDOTINIT, np.arange(len_b) + len_n)
        self._register_sparse(HydVarEq.MDOTSLACKINIT, len_n, self.slack_nodes, slack_vals)

        self._register(HydVarEq.NODE,   np.arange(len_n))
        self._register(HydVarEq.BRANCH, np.arange(len_b) + len_n)
        self._register_sparse(HydVarEq.SLACK, len_n, self.slack_nodes, slack_vals)
```

Every hydraulic unknown/equation type gets a contiguous block of matrix indices. A component asks
for "the column for `PINIT` at these node rows" via `sys_idx.idx(HydVarEq.PINIT, some_node_indices)`
- it never computes a matrix index by hand.

`MDOTSLACKINIT`/`SLACK` only exist at P-type (slack) nodes, a small subset of all nodes - but
`_register_sparse` still sizes their block like the **full** node array, with `-1` at every
non-slack position:

```python
def _register_sparse(self, key, full_size, node_indices, values):
    arr = np.full(full_size, -1, dtype=np.int32)
    arr[node_indices] = values
    self._blocks[self._block_key(key)] = arr
    if len(values):
        self._size = max(self._size, int(values.max()) + 1)
```

That means `sys_idx.idx(HydVarEq.MDOTSLACKINIT, eg_nodes)` works with raw node indices exactly like
`PINIT` does - no separate "rank within the slack subset" translation needed by callers. (This used
to require `np.searchsorted(slack_nodes, eg_nodes)` in every caller; folding the sparse layout into
`_register_sparse` once removed that boilerplate everywhere it was needed.)

One consequence worth knowing if you call `sys_idx.idx(key)` **without** a subset: for a sparse
block, that returns the raw, `-1`-padded, full-`len_n`-sized array, not a compact "just the slack
values" array - anyone doing this (e.g. applying the Newton update for *all* slack nodes at once)
must pass `slack_nodes` explicitly as the subset, not rely on the bare block.

`_block_key` exists purely so a subclass can namespace keys without re-implementing `idx`/
`_register`/`_register_sparse` three times: `combined_pipeflow`'s `HydThermSystemIndex` combines
hydraulic and thermal blocks in one system, where `HydVarEq.NODE` and `ThermVarEq.NODE` are equal
as plain strings (both enums subclass `str`) and would otherwise collide as dict keys. It overrides
just `_block_key` to key on `(type(var), var)` instead, and every read/write path picks that up
automatically.

## 4. Phase 2 - the linear system: `register_hydraulic_equations`

`register_hydraulic_equations` runs on every component, every Newton iteration, collecting linear
contributions (Jacobian entries + load/residual values) into a `ComponentRegistry`, which then gets
assembled into one sparse matrix and solved for the whole net.

### `ExtGrid.register_hydraulic_equations`

```python
@classmethod
def register_hydraulic_equations(cls, net, branch_pit, node_pit, sys_idx, registry):
    # register only for nodes that actually have an active ext_grid row - NOT every P-type
    # node in the system (a circ_pump also marks its own flow junction as NODE_TYPE=P purely
    # to anchor a pressure reference; that node is none of ExtGrid's business)
    ext_grids = net[cls.table_name()]
    ext_grids = ext_grids[ext_grids[cls.active_identifier()].values]
    p_grids = ext_grids[np.isin(ext_grids.type.values, ["p", "pt"])]
    if not len(p_grids):
        return

    junction_lookup = get_lookup(net, "node", "index_active_hydraulics")[
        cls.get_connected_node_type().table_name()]
    # one entry per ext_grid ROW - deliberately NOT deduplicated by node
    eg_nodes = junction_lookup[p_grids[cls.get_node_col()].values].astype(np.int32)
    eg_nodes = eg_nodes[eg_nodes != -1]   # drop disconnected
    if not len(eg_nodes):
        return

    p_col     = sys_idx.idx(HydVarEq.PINIT,         eg_nodes)
    slack_col = sys_idx.idx(HydVarEq.MDOTSLACKINIT, eg_nodes)
    slack_eq  = sys_idx.idx(HydVarEq.SLACK,         eg_nodes)
    n_eq      = sys_idx.idx(HydVarEq.NODE,          eg_nodes)

    # SLACK row: pressure fix, δPINIT = 0 - MEAN (see below)
    registry.add(ComponentEquations(
        rows=slack_eq.astype(np.int32), cols=p_col.astype(np.int32),
        data=np.ones(len(slack_eq), dtype=np.float64),
        load_rows=slack_eq.astype(np.int32), load_data=np.zeros(len(slack_eq), dtype=np.float64),
        mode=EqWriteMode.MEAN,
    ))

    # NODE row: MDOTSLACKINIT joins the mass balance, free to absorb residual - ADDITIVE (default)
    registry.add(ComponentEquations(
        rows=n_eq.astype(np.int32), cols=slack_col.astype(np.int32),
        data=np.ones(len(n_eq), dtype=np.float64),
        load_rows=n_eq.astype(np.int32),
        load_data=node_pit[eg_nodes, IdxNode.MDOTSLACKINIT].astype(np.float64),
    ))
```

Two non-obvious things about this code, both learned the hard way while building it:

**(a) `"index_active_hydraulics"`, never the plain `"index"` lookup, in this phase.**
`register_pit_node_entries` runs *before* connectivity reduction, against the full node array, so
the plain `"index"` lookup is correct there. `register_hydraulic_equations` runs *after*
reduction, against the smaller *active* pit - using the plain lookup here indexes into the
wrong-sized array and either crashes outright (`IndexError: index 9 is out of bounds for axis 0
with size 6`) or, worse, silently resolves to the wrong node. Every hydraulic-phase lookup needs
the `"index_active_hydraulics"` variant (there's a matching `"index_active_heat_transfer"` for
`register_thermal_equations`). `-1` in the lookup means "disconnected, dropped from the active
pit" - filter those out (`eg_nodes[eg_nodes != -1]`) rather than passing them through.

**(b) Registering once per ext_grid *row*, not once per de-duplicated node, is what makes "multiple
ext_grids share one junction" work correctly without any extra code.** If two ext_grids sit at the
same junction, this method's row-building logic runs its per-row arrays with that junction's node
index appearing *twice*, and both entries target the *same* matrix row/column. For the `SLACK`
row, two registrations land on the same `(row, col)` - which is exactly why it must be `MEAN`, not
`UNIQUE`: `UNIQUE` conflict-checks new rows against rows *other* registered entries already claim,
and duplicate rows landing there would raise. For the `NODE` row (`ADDITIVE`, the default), two
contributions to the same `(row, col)` don't conflict - they *sum* when the sparse matrix is built
(`scipy.sparse.csr_matrix` sums duplicate COO entries), so the row's coefficient on
`MDOTSLACKINIT` becomes `N` for `N` co-located ext_grids. Given the node's mass balance forces
`N * MDOTSLACKINIT = (total residual left over by everything else)`, Newton solves directly for
each ext_grid's own share - `extract_results` just reads `node_pit[eg_nodes, MDOTSLACKINIT]`
per row, no separate division needed. That's exactly the `net2.res_ext_grid` output you saw
above: `-2.5` for each of the two co-located ext_grids, straight out of the solve.

## 5. Assembly and solve

`solve_hydraulics` (`pf/calculation.py`) ties every component's registrations into one solve:

```python
sys_idx = HydraulicSystemIndex(node_pit, branch_pit)
eq_registry = ComponentRegistry()

for comp in net['component_list']:
    comp.register_hydraulic_equations(net, branch_pit, node_pit, sys_idx, eq_registry)

sz = sys_idx.size()
rows, cols, data, epsilon = eq_registry.assemble(sz)
jacobian = csr_matrix((data, (rows, cols)), shape=(sz, sz))

x = spsolve(jacobian, epsilon)

branch_pit[:, IdxBranch.MDOTINIT] -= x[mdot_idx] * options["alpha"]
node_pit[:, IdxNode.PINIT]        -= x[p_idx]    * options["alpha"]
node_pit[slack_nodes, IdxNode.MDOTSLACKINIT] -= x[msl_idx]
```

`ComponentRegistry.assemble()` (`pf/system_index.py`) is where `EqWriteMode` actually gets acted
on: `UNIQUE`-tagged entries registered via `add_override` strip any competing `ADDITIVE`
contributions to the same row before concatenation; `MEAN`-tagged entries (from either bucket) are
pulled into a separate pool, grouped by `(row, col)`, averaged (Jacobian data summed then divided
by count; load values NaN-filtered, summed, divided by count), and everything else is just
concatenated and left for `csr_matrix` to sum on construction.

Nothing here is component-specific - `solve_hydraulics` doesn't know or care that `ExtGrid` exists;
it only knows "call `register_hydraulic_equations` on everything, assemble what comes back, solve."
That's the entire point of the refactor away from the old `build_system_matrix.py` pattern: each
component owns its own contribution to the system, described declaratively (rows/cols/data +
write-mode), instead of every component needing to know how to poke directly into one big,
centrally-assembled matrix-building function.

## 6. Cross-component coordination without tight coupling: the `VAR_MASS_SLACK` case study

`ExtGrid` isn't the only component that marks a node `NODE_TYPE = P`. `CircPump` (both
`CirculationPumpMass` and `CirculationPumpPressure`, via the shared `CirculationPump` base class in
`abstract_models/circulation_pump.py`) marks its own flow junction `NODE_TYPE = P` too - not because
it's an external mass source, but purely to **anchor an absolute pressure reference**: without at
least one such anchor somewhere in the network, pressure is only ever defined up to an arbitrary
additive constant (branch/momentum equations only constrain pressure *differences*), and the
system would be singular.

That distinction - "P-type node with a genuine external mass connection" vs. "P-type node that's
just a pressure anchor" - isn't visible from `NODE_TYPE` alone. Both look identical to
`HydraulicSystemIndex`, which builds `slack_nodes` purely from `NODE_TYPE == P` regardless of which
component set it. Left alone, a circ-pump-only node would get exactly the same treatment as a real
ext_grid: `MDOTSLACKINIT` free to absorb whatever residual mass the rest of the network leaves
over - silently hiding a genuine supply/demand mismatch in a network that has no real external
connection to explain it.

Let's see this concretely: a circulation pump only (no ext_grid at all), with a sink and source
that deliberately don't balance (3 kg/s drawn, only 2 kg/s supplied):

In [ ]:
from pandapipes.pf.pipeflow_setup import PipeflowNotConverged

def build_circ_pump_net(sink, source):
    net = pandapipes.create_empty_network("net", add_stdtypes=False)
    j1 = pandapipes.create_junction(net, pn_bar=5, tfluid_k=283.15)
    j2 = pandapipes.create_junction(net, pn_bar=5, tfluid_k=283.15)
    j3 = pandapipes.create_junction(net, pn_bar=5, tfluid_k=283.15)
    j4 = pandapipes.create_junction(net, pn_bar=5, tfluid_k=283.15)
    pandapipes.create_pipe_from_parameters(net, j1, j2, k_mm=1., length_km=0.4338, inner_diameter_mm=102.2)
    pandapipes.create_pipe_from_parameters(net, j3, j4, k_mm=1., length_km=0.2637, inner_diameter_mm=102.2)
    pandapipes.create_circ_pump_const_mass_flow(net, j4, j1, 5, 5, 300, type="pt")
    pandapipes.create_heat_exchanger(net, j2, j3, qext_w=200000, inner_diameter_mm=100)
    pandapipes.create_sink(net, j1, sink)
    pandapipes.create_source(net, j4, source)
    pandapipes.create_fluid_from_lib(net, "water", overwrite=True)
    return net, j1

# UNBALANCED: sink (3 kg/s) != source (2 kg/s), and there is no real ext_grid anywhere
net_bad, j1 = build_circ_pump_net(sink=3, source=2)
try:
    pandapipes.pipeflow(net_bad, mode="sequential")
    print("converged:", net_bad.converged, "<- this would be a silent bug")
except PipeflowNotConverged:
    print("Correctly raised PipeflowNotConverged: the network is genuinely inconsistent")


In [ ]:
# BALANCED: sink == source -> should converge, and MDOTSLACKINIT at the pump's anchor node
# should be exactly 0 (it's not a real mass source, just a pressure reference)
from pandapipes.idx_node import IdxNode as IdxNodeCheck

net_ok, j1_ok = build_circ_pump_net(sink=2, source=2)
pandapipes.pipeflow(net_ok, mode="sequential")
print("converged:", net_ok.converged)
print("MDOTSLACKINIT at the circ_pump's anchor node (expect 0.0):",
      net_ok["_pit"]["node"][j1_ok, IdxNodeCheck.MDOTSLACKINIT])


`VAR_MASS_SLACK` is the flag that lets these two components coordinate without one having to know
the other's internals: `ExtGrid.register_pit_node_entries` sets it (`UNIQUE`, see above) for every
node with a real ext_grid; `CircPump._register_slack_equations` reads it to decide how to treat
`MDOTSLACKINIT` at its own flow junction:

```python
@classmethod
def _register_slack_equations(cls, net, node_pit, sys_idx, registry):
    ...
    junction_lookup = get_lookup(net, "node", "index_active_hydraulics")[
        cls.get_connected_node_type().table_name()]
    pump_nodes = junction_lookup[p_pumps[tn_col].values].astype(np.int32)
    pump_nodes = pump_nodes[pump_nodes != -1]
    if not len(pump_nodes):
        return

    # pressure fix - always, for every own flow junction (MEAN: coexists with ExtGrid's own
    # pressure fix if a real ext_grid happens to sit at the same node too)
    p_col = sys_idx.idx(HydVarEq.PINIT, pump_nodes)
    slack_eq = sys_idx.idx(HydVarEq.SLACK, pump_nodes)
    registry.add(ComponentEquations(
        rows=slack_eq.astype(np.int32), cols=p_col.astype(np.int32),
        data=np.ones(len(slack_eq), dtype=np.float64),
        load_rows=slack_eq.astype(np.int32), load_data=np.zeros(len(slack_eq), dtype=np.float64),
        mode=EqWriteMode.MEAN,
    ))

    # where there's no real ext_grid at this node (VAR_MASS_SLACK == 0), reset MDOTSLACKINIT to 0
    # before it's read as this iteration's Newton seed, then add it into the node's balance
    # exactly like ExtGrid would (plain add(), ADDITIVE - joins the genuine pipe/sink balance,
    # does not replace or strip it)
    force_zero = np.unique(pump_nodes[node_pit[pump_nodes, IdxNode.VAR_MASS_SLACK] == 0])
    node_pit[force_zero, IdxNode.MDOTSLACKINIT] = 0.

    n_eq = sys_idx.idx(HydVarEq.NODE, pump_nodes)
    slack_col = sys_idx.idx(HydVarEq.MDOTSLACKINIT, pump_nodes)
    registry.add(ComponentEquations(
        rows=n_eq.astype(np.int32), cols=slack_col.astype(np.int32),
        data=np.ones(len(n_eq), dtype=np.float64),
        load_rows=n_eq.astype(np.int32),
        load_data=node_pit[pump_nodes, IdxNode.MDOTSLACKINIT].astype(np.float64),  # == 0. now
    ))
```

The pressure fix is registered unconditionally, `MEAN`, for exactly the same reason as `ExtGrid`'s
own - so it peacefully coexists with `ExtGrid`'s own pressure-fix row if a real ext_grid happens to
sit at the same junction too, instead of a `UNIQUE`-vs-`UNIQUE` conflict.

The `MDOTSLACKINIT` handling is the interesting part, and got one thing wrong before landing on
this shape - worth knowing, since it's an easy mistake to repeat:

- **First (broken) attempt:** replace the node's entire balance row with `MDOTSLACKINIT = 0` via
  `add_override(..., mode=UNIQUE)`. This *does* force `MDOTSLACKINIT` to 0, but `UNIQUE`-as-override
  strips **every** other contribution to that row too - including the genuine pipe/sink mass
  balance other components additively contribute there. Result: the node's real demand silently
  stopped being enforced at all, and the unbalanced example above would have *converged* to a
  wrong answer (the sink's extra 1 kg/s just vanishing) instead of correctly failing. The lesson:
  `UNIQUE`/`add_override` is for "this row is exclusively mine", not for "let me also pin one more
  thing using a row something else already needs."
- **Working version (above):** reset `MDOTSLACKINIT` to `0.` directly in `node_pit` *before*
  building the equation, then add it into the balance the same way `ExtGrid` does (`ADDITIVE`, not
  replacing anything). Since `register_hydraulic_equations` runs fresh every Newton iteration, the
  reset happens every iteration, right before the residual for that row is computed - so
  `MDOTSLACKINIT` never gets the chance to settle on a nonzero value that would silently absorb a
  real imbalance. If the surrounding network genuinely doesn't balance, this residual simply never
  reaches zero and `pipeflow()` correctly raises `PipeflowNotConverged` instead of reporting a
  wrong success - exactly what the cell above demonstrated.

The broader point: two components can coordinate through a **shared PIT flag**, each only reading/
writing the parts of `node_pit` relevant to itself, without either one needing to import the other
or know about its internal logic. `ExtGrid` doesn't know `CircPump` exists; `CircPump` only cares
whether *some* component already claimed `VAR_MASS_SLACK` at its own node.

## 7. Checklist: adding a new component

- **Table & PIT columns:** does your component need new `IdxNode`/`IdxBranch` columns? Add them at
  the end of the respective file and bump `node_cols`/`branch_cols`. Don't reuse another
  component's column for an unrelated purpose - if two components need the same *kind* of flag
  (like `VAR_MASS_SLACK`), that's a sign it should be a shared, well-named column, not overloaded.
- **`register_pit_node_entries`/`register_pit_branch_entries`:** write your component's PIT data
  here, using `add()` for baseline values and `add_override()` when you need to win over another
  component's default. Pick `PitWriteMode` by what the *value itself* means: `UNIQUE` for "exactly
  one true value, conflict if two different sources disagree without a mechanism to merge them",
  `MEAN` for "several sources may legitimately target the same cell and averaging is a sane
  fallback", `ADDITIVE` for genuine sums/counts.
- **`register_hydraulic_equations`/`register_thermal_equations`:** always use
  `"index_active_hydraulics"`/`"index_active_heat_transfer"` lookups here, never the plain
  `"index"` one - the plain one is for the pre-reduction PIT-fill phase only. Filter out `-1`
  (disconnected) before using indices further.
- **Choosing `EqWriteMode`:** `ADDITIVE` (default) for anything that should sum with other
  components' contributions to the same row (the normal case - e.g. every branch's contribution to
  its endpoint nodes' mass balance). `MEAN` when multiple components might legitimately target the
  same row with independent "this is the target value" claims (pressure/temperature fixes).
  `UNIQUE` only for a row that is genuinely, exclusively yours - and even then, prefer `add()` +
  `ADDITIVE` if you're only *adding* a term, reserving `add_override()` + `UNIQUE` for when you
  deliberately want to override/replace whatever else targets that row (which almost always means
  you're accepting that anything else contributing there gets silently dropped - make sure that's
  really what you want, see the first-attempt mistake in §6).
- **Needing to coordinate with another component without importing it:** a shared PIT flag column,
  written by whichever component has the authoritative answer and read by whichever needs to adapt
  its own behavior, is the established pattern (`VAR_MASS_SLACK` being the worked example here).
- **Test both the isolated case and the coexistence case:** if your component can end up at the
  same node as another P-type-marking component (ext_grid + circ_pump being the concrete example),
  write a test for that combination specifically - it's exactly the kind of interaction that looks
  fine in isolation and breaks silently in combination.